# ForestWatch Papua — Training Spesifik Wilayah **Merauke + Boven Digoel**
### Model 1: Attention U-Net (SCSE) + ResNet-50 encoder (ImageNet)

Notebook **mandiri & end-to-end** untuk satu wilayah krisis deforestasi: *preprocessing →
EDA distribusi piksel (teks) → bundling → training → evaluasi → ONNX*. Semua data & artefak
disimpan di satu folder Drive khusus (`Training_Merauke_Boven_Digoel/`).

**Kenapa Merauke + Boven Digoel?** (data 2024–2025)
- **Merauke PSN food/sugarcane estate = driver deforestasi primer #1 Indonesia 2024**
  (22.272 ha ekosistem alami dibuka Jan'24–Jun'25; 9.835 ha hutan primer).
- **Boven Digoel** = hotspot sawit proyek Tanah Merah (≥3.700 ha hutan primer hilang).
- Kedua kabupaten **padat sinyal transisi Hutan→Lahan Terbuka / Sawit / Pertanian** →
  ideal untuk model belajar deforestasi pada dataset terfokus & kecil.

**Kenapa Attention U-Net (SCSE)?** (dukungan literatur)
- Attention gate pada skip-connection menekan aktivasi tak relevan & menonjolkan region
  informatif — penting untuk citra Sentinel-2 yang spektralnya tumpang-tindih antar-kelas.
- Blok **spatial-and-channel Squeeze-and-Excitation (scSE)** merekalibrasi fitur spasial &
  spektral secara adaptif.
- Brown et al. (2022), *"An attention-based U-Net for detecting deforestation within satellite
  sensor imagery"* (ISPRS / Int. J. Applied Earth Obs.) melaporkan F1 piksel 0.946–0.977 untuk
  deteksi deforestasi Sentinel-2 dengan Attention U-Net.
- *Attention-Based Semantic Segmentation Networks for Forest Applications* (Forests, 2023, MDPI)
  menegaskan keunggulan mekanisme attention untuk monitoring deforestasi berbasis penginderaan jauh.

**Cakupan tile (grid 6×6 atas bbox Papua):** `{24, 25, 26, 30, 31, 32}` — kolom lon 137.5–141.2,
lat -9.5 to -4.5 (Merauke + Boven Digoel; `idx = i*6 + j`, i=kolom lon, j=baris lat).

> **Catatan kejujuran data:** laporkan mIoU/akurasi **apa adanya** untuk wilayah ini. Lebih baik
> mIoU 0,68 yang nyata daripada 0,90 yang fiktif (sesuai pedoman ForestWatch & kriteria Substansi & Data).

## Bagian 0 — Setup environment (Colab / Lab)

- **Google Colab** → `ENV = "colab"`: clone repo + install package + mount Drive.
- **PC lab + Drive Desktop (mode Mirror)** → `ENV = "lab"`: sesuaikan `DRIVE_ROOT`, jalankan
  `pip install -e ".[ml]"` sekali di clone repo lokal.

Prasyarat: patch Papua se-36-tile sudah ada di `ForestWatch_Patches/tile_XXX/` di Drive (hasil
Bagian 1–14 notebook pipeline utama).

In [ ]:
# === Bagian 0 — Setup (set ENV = "colab" / "lab") ===
ENV = "colab"   # "colab" (Google Colab) | "lab" (PC + Drive Desktop mode Mirror)
from pathlib import Path

DRIVE_ROOT = None

if ENV == "colab":
    import subprocess, sys, importlib
    subprocess.run(
        "cd /content && (git -C fw_repo pull -q || git clone --depth 1 "
        "https://github.com/Ridho-Dwi-Syahputra/forestwatch-model.git fw_repo)",
        shell=True, check=False,
    )
    subprocess.run("pip install -q -e /content/fw_repo[gee,gis,ml]", shell=True, check=False)
    if "/content/fw_repo/src" not in sys.path:
        sys.path.insert(0, "/content/fw_repo/src")
    for _m in [m for m in list(sys.modules) if m == "forestwatch" or m.startswith("forestwatch.")]:
        del sys.modules[_m]
    importlib.invalidate_caches()
    from google.colab import drive
    drive.mount("/content/drive")
    DRIVE_ROOT = Path("/content/drive/MyDrive/Satria Data 3.0")
elif ENV == "lab":
    DRIVE_ROOT = Path(r"G:/My Drive/Satria Data 3.0")   # <- SESUAIKAN path mount Drive Desktop lab
else:
    raise ValueError("ENV harus 'colab' atau 'lab'")

assert DRIVE_ROOT.exists(), f"DRIVE_ROOT {DRIVE_ROOT} tidak ada — cek mount Drive."

import torch
_gpu = f" ({torch.cuda.get_device_name(0)})" if torch.cuda.is_available() else ""
print(f"ENV={ENV} | DRIVE_ROOT={DRIVE_ROOT} | CUDA={torch.cuda.is_available()}{_gpu}")

In [ ]:
# === Path sumber + folder khusus wilayah + konfigurasi tile ===
import json
from forestwatch.config import load_config
from forestwatch.constants import N_CLASSES, CLASS_NAMES
from forestwatch.utils.io import save_json, load_json
cfg = load_config()

# --- Sumber patch (se-Papua, 36 tile) ---
PATCH_DIR         = DRIVE_ROOT / 'ForestWatch_Patches'            # papua/tile_XXX/p*.npz
PATCHES_TRANSFER  = DRIVE_ROOT / 'ForestWatch_Patches_Transfer'   # kelas langka dari wilayah lain
AUGMENTED_PATCHES = DRIVE_ROOT / 'Augmented_Patches'              # augmentasi offline kelas minor

# --- Folder khusus wilayah: SEMUA data & artefak Merauke+BD disimpan di sini ---
AREA_DIR  = DRIVE_ROOT / 'Training_Merauke_Boven_Digoel'
EDA_DIR   = AREA_DIR / 'eda_cache'
MODEL_DIR = AREA_DIR / 'model_1_attention_unet'
for d in (AREA_DIR, EDA_DIR, MODEL_DIR):
    d.mkdir(parents=True, exist_ok=True)

# --- Whitelist tile Merauke + Boven Digoel (grid 6x6, idx = i*6 + j) ---
TILE_WHITELIST = {24, 25, 26, 30, 31, 32}

print('AREA_DIR       :', AREA_DIR)
print('TILE_WHITELIST :', sorted(TILE_WHITELIST))
print('Resep training :', cfg['training']['loss']['type'],
      '| epochs:', cfg['training']['epochs'], '| batch:', cfg['training']['batch_size'])

In [ ]:
# === Preprocessing 1: filter geografis Merauke+BD + split holdout ===
from forestwatch.data import list_patches, split_files

def _tile_idx(f):
    # PATCH_DIR/tile_030/p*.npz -> 30 (atau -1 bila bukan subfolder tile)
    name = Path(f).parent.name
    return int(name.split("_")[-1]) if name.startswith("tile_") else -1

# Patch Papua DIFILTER ke wilayah; transfer + aug DIPERTAHANKAN (bantu kelas langka di TRAIN saja).
papua_all      = list_patches(PATCH_DIR)
papua_files    = [f for f in papua_all if _tile_idx(f) in TILE_WHITELIST]
transfer_files = list_patches(PATCHES_TRANSFER)
aug_files      = list_patches(AUGMENTED_PATCHES)
assert papua_files, (
    f"Tidak ada patch Merauke+BD di {PATCH_DIR} untuk tile {sorted(TILE_WHITELIST)}. "
    "Cek nama subfolder tile_XXX & sync Drive."
)

# Distribusi per-tile (transparansi cakupan)
from collections import Counter
_per_tile = Counter(_tile_idx(f) for f in papua_files)
print("Patch papua per-tile (Merauke + Boven Digoel):")
for ti in sorted(TILE_WHITELIST):
    print(f"  tile_{ti:03d}: {_per_tile.get(ti, 0):>7,} patch")

# Split sumber-aware (seed=42): val/test = holdout MURNI Merauke+BD; train + transfer + aug.
train_p, val_p, test_p = split_files(papua_files, train_ratio=0.8, val_ratio=0.1, seed=42)
final_train_files = list(train_p) + list(transfer_files) + list(aug_files)
print(f"\npapua(Merauke+BD)={len(papua_files):,} dari {len(papua_all):,} total se-Papua")
print(f"train={len(final_train_files):,} (papua={len(train_p):,}+transfer={len(transfer_files):,}"
      f"+aug={len(aug_files):,}) | val={len(val_p):,} | test={len(test_p):,} (holdout Merauke+BD)")

In [ ]:
# === Preprocessing 2: scan distribusi piksel per patch (cached -> skip jika sudah ada) ===
import numpy as np
from concurrent.futures import ThreadPoolExecutor, as_completed
from tqdm.auto import tqdm

SCAN_CACHE = EDA_DIR / 'patch_class_counts.json'

def _count_patch(path):
    lab = np.load(path)['lab'].flatten().astype(int)
    return np.bincount(lab, minlength=N_CLASSES).tolist()

def scan_patches(files, cache_path, max_workers=64):
    cache_p = Path(cache_path)
    dist = {}
    if cache_p.exists():
        dist = json.loads(cache_p.read_text(encoding='utf-8'))
        missing = [f for f in files if str(f) not in dist]
        if not missing:
            print(f"[cache] {cache_p.name}: semua {len(files):,} patch sudah tercache.")
            return dist
        print(f"[cache] {len(missing):,} patch baru, scan sekarang ...")
        files = missing
    with ThreadPoolExecutor(max_workers=max_workers) as exe:
        futs = {exe.submit(_count_patch, f): str(f) for f in files}
        for fut in tqdm(as_completed(futs), total=len(futs), desc='Scan piksel', unit='patch'):
            dist[futs[fut]] = fut.result()
    cache_p.write_text(json.dumps(dist), encoding='utf-8')
    print(f"[cache] disimpan -> {cache_p}")
    return dist

# Scan SEMUA file yang dipakai (train final + val + test) sekali, simpan di folder wilayah.
patch_dist = scan_patches(list(final_train_files) + list(val_p) + list(test_p), SCAN_CACHE)
print(f"Total patch tercatat: {len(patch_dist):,}")

In [ ]:
# === EDA: deskripsi distribusi sebaran piksel per kelas (TEKS) ===
def _agg(files):
    px  = [0] * N_CLASSES
    dom = [0] * N_CLASSES
    for f in files:
        c = patch_dist.get(str(f), [0] * N_CLASSES)
        for k in range(N_CLASSES):
            px[k] += c[k]
        dom[int(np.argmax(c))] += 1
    return px, dom

def eda_report(files, title):
    px, dom = _agg(files)
    grand = sum(px) or 1
    nz = [px[c] for c in range(N_CLASSES) if px[c] > 0]
    imb = (max(nz) / min(nz)) if nz else float('nan')
    print('=' * 78)
    print(f"  {title}")
    print(f"  Total patch : {len(files):,}   |   Total piksel : {grand:,}")
    print('=' * 78)
    print(f"  {'Kelas':<16}{'Piksel':>18}{'%':>9}{'Patch dominan':>16}")
    print('  ' + '-' * 60)
    for c in range(N_CLASSES):
        print(f"  {CLASS_NAMES[c]:<16}{px[c]:>18,}{100*px[c]/grand:>8.2f}%{dom[c]:>16,}")
    print('  ' + '-' * 60)
    print(f"  {'TOTAL':<16}{grand:>18,}{100.0:>8.2f}%{len(files):>16,}")
    print(f"  Rasio ketidakseimbangan (maks/min piksel kelas != 0): {imb:,.1f}x")
    print('=' * 78)
    return px

print("DESKRIPSI DISTRIBUSI SEBARAN PIKSEL — Wilayah Merauke + Boven Digoel\n")
px_train = eda_report(final_train_files, "TRAIN FINAL (Merauke+BD + transfer + aug)")
px_val   = eda_report(val_p,   "VAL (holdout Merauke+BD)")
px_test  = eda_report(test_p,  "TEST (holdout Merauke+BD)")

# Interpretasi naratif (teks) — bantu penulisan esai.
_tr = sum(px_train) or 1
_dom_c = int(np.argmax(px_train)); _rare_c = int(np.argmin([p if p > 0 else 1e18 for p in px_train]))
print("\nInterpretasi:")
print(f"- Kelas dominan TRAIN: {CLASS_NAMES[_dom_c]} ({100*px_train[_dom_c]/_tr:.1f}% piksel) — "
      "wajar untuk wilayah ini (pesisir/hutan luas).")
print(f"- Kelas terlangka TRAIN: {CLASS_NAMES[_rare_c]} ({100*px_train[_rare_c]/_tr:.2f}% piksel) — "
      "ditangani median-frequency class weights + weighted sampler saat training.")
print("- Val/Test sengaja MURNI Merauke+BD (tanpa transfer/aug) agar metrik jujur utk wilayah ini.")

In [ ]:
# === Preprocessing 3 (OPSIONAL): subsample patch dominan agar lebih seimbang ===
# Default OFF (dataset wilayah sudah kecil; imbalans ditangani class weights + sampler).
# Nyalakan bila EDA menunjukkan satu kelas (mis. Perairan) sangat mendominasi (> ~70%).
import random

SUBSAMPLE_ENABLED = False
PIXEL_TARGETS = {0: 400_000_000, 1: 500_000_000}   # {class_id: target piksel} bila ENABLED

def subsample_to_targets(files, targets, seed=42):
    rng = random.Random(seed)
    groups = {c: [] for c in range(N_CLASSES)}
    for f in files:
        groups[int(np.argmax(patch_dist.get(str(f), [0]*N_CLASSES)))].append(f)
    selected = []
    for cls in range(N_CLASSES):
        g = list(groups[cls])
        if cls not in targets:
            selected.extend(g); continue
        rng.shuffle(g); tot, kept = 0, []
        for f in g:
            tot += patch_dist.get(str(f), [0]*N_CLASSES)[cls]; kept.append(f)
            if tot >= targets[cls]:
                break
        selected.extend(kept)
        print(f"  {CLASS_NAMES[cls]:<16}: {len(g):,} -> {len(kept):,} patch ({tot:,} px)")
    return selected

if SUBSAMPLE_ENABLED:
    print("Subsample AKTIF:")
    selected_train = subsample_to_targets(final_train_files, PIXEL_TARGETS)
    print(f"\nTRAIN: {len(final_train_files):,} -> {len(selected_train):,} patch")
    eda_report(selected_train, "TRAIN setelah subsample")
else:
    selected_train = list(final_train_files)
    print(f"Subsample NONAKTIF — pakai semua {len(selected_train):,} patch train apa adanya.")

In [ ]:
# === Preprocessing 4: hitung class weights (median-frequency) dari TRAIN terpilih ===
from forestwatch.training.metrics import median_frequency_weights

dist_train = [0] * N_CLASSES
for f in selected_train:
    c = patch_dist.get(str(f), [0]*N_CLASSES)
    for k in range(N_CLASSES):
        dist_train[k] += c[k]

class_weights = median_frequency_weights(dict(enumerate(dist_train)), n_classes=N_CLASSES)
print("Class weights (median-frequency, dari TRAIN Merauke+BD):")
for c in range(N_CLASSES):
    print(f"  {CLASS_NAMES[c]:<16} piksel={dist_train[c]:>16,}  weight={class_weights[c]:>8.4f}")
save_json({'class_weights': [float(w) for w in class_weights]}, AREA_DIR / 'class_weights.json')
print(f"\nDisimpan -> {AREA_DIR / 'class_weights.json'}")

In [ ]:
# === Preprocessing 5: bundle .tar + sampler cache ke folder wilayah (idempoten) ===
from forestwatch.data.dataset import create_dataset_archives, compute_patch_sampler_weights

def _arcname(f):
    f = Path(f)
    for root, pfx in [(PATCH_DIR, 'papua'), (PATCHES_TRANSFER, 'transfer'), (AUGMENTED_PATCHES, 'aug')]:
        try:
            return f"{pfx}/{f.relative_to(root).as_posix()}"
        except ValueError:
            continue
    return f.name

sel_set = set(_arcname(f) for f in selected_train)
train_items = [(_arcname(f), f) for f in selected_train]
val_items   = [(_arcname(f), f) for f in val_p]
test_items  = [(_arcname(f), f) for f in test_p]

splits = {'train': train_items, 'val': val_items, 'test': test_items}
create_dataset_archives(splits, AREA_DIR, n_train_parts=4, max_workers=64)

# Daftar arcname train terpilih (dipakai notebook lain / audit).
save_json(sorted(sel_set), AREA_DIR / 'selected_train_patches.json')

# Sampler cache SHARED — key = parts[-3:] dari arcname berprefiks 'train/' (cocok dgn key dari
# path lokal hasil extract_dataset_archives). Idempoten: skip bila sudah ada.
SAMPLER_CACHE = AREA_DIR / 'patch_sampler_weights_shared.json'
if SAMPLER_CACHE.exists():
    print(f"[skip] {SAMPLER_CACHE.name} sudah ada.")
else:
    sampler_files = [src for _, src in train_items]
    sampler_keys  = ["/".join((Path('train') / arc).parts[-3:]) for arc, _ in train_items]
    compute_patch_sampler_weights(sampler_files, class_weights, cache_path=SAMPLER_CACHE, keys=sampler_keys)
    print(f"[ok] {SAMPLER_CACHE.name} dihitung.")
print(f"\nSemua bahan training tersimpan di: {AREA_DIR}")

In [ ]:
# === Preprocessing 6: ekstrak .tar ke disk LOKAL (I/O cepat, lepas dari Drive FUSE) ===
from forestwatch.data.dataset import extract_dataset_archives

LOCAL_DIR = (Path('/content/merauke_local') if ENV == 'colab' else Path.home() / 'merauke_local')
local_dirs = extract_dataset_archives(AREA_DIR, LOCAL_DIR, max_workers=64)

final_train_files = list_patches(local_dirs['train'])
val_files  = list_patches(local_dirs['val'])
test_files = list_patches(local_dirs['test'])
# Key sampler dari path lokal = parts[-3:] -> cocok dgn cache (sampler_keys=None aman).
print(f"[lokal] train={len(final_train_files):,} val={len(val_files):,} test={len(test_files):,} -> {LOCAL_DIR}")

## Bagian Training — Attention U-Net (SCSE) + ResNet-50

Resep identik konfigurasi proyek (`configs/default.yaml`): loss `focal_tversky`, median-frequency
class weights, weighted sampler kelas langka, AMP, warmup→cosine LR, transfer-learning 2-tahap
(freeze encoder beberapa epoch awal), grad-clip, resume dari checkpoint. Artefak → `MODEL_DIR`.

In [ ]:
# === Worker tuning ===
import os
N_WORKERS = max(2, min(8, (os.cpu_count() or 2) - 1))
print(f"os.cpu_count()={os.cpu_count()} -> num_workers={N_WORKERS} "
      "(persistent_workers=True, pin_memory di build DataLoader).")

In [ ]:
# === Build DataLoaders + model Attention U-Net (SCSE) + loss ===
from forestwatch.data import build_dataloaders_from_files
from forestwatch.model.architecture import build_unet, count_parameters
from forestwatch.model.losses import make_loss_fn

MODEL_KEY  = "model_1_attention_unet"
MODEL_ARCH = dict(architecture="unet_scse", encoder_name="resnet50")
CKPT_PATH  = MODEL_DIR / 'best_model.pt'

use_sampler = cfg['training'].get('use_weighted_sampler', True)
train_loader, val_loader, test_loader = build_dataloaders_from_files(
    final_train_files, val_files, test_files,
    batch_size=cfg['training']['batch_size'], num_workers=N_WORKERS,
    augment_p=cfg['training']['augmentation'],
    class_weights=class_weights if use_sampler else None,
    sampler_cache=SAMPLER_CACHE,   # cache shared di folder wilayah
    sampler_keys=None,             # file lokal -> key parts[-3:] cocok dgn cache
    persistent_workers=True,
)
print(f"Train {len(train_loader.dataset)} | Val {len(val_loader.dataset)} | Test {len(test_loader.dataset)}")

model = build_unet(
    in_channels=cfg['model']['in_channels'], classes=cfg['model']['classes'],
    encoder_weights=cfg['model']['encoder_weights'], **MODEL_ARCH,
)
print(f"{MODEL_KEY}: {count_parameters(model):,} param trainable")

lc = cfg['training']['loss']
loss_fn = make_loss_fn(
    loss_type=lc['type'],
    class_weights=class_weights if cfg['training'].get('use_class_weights', True) else None,
    tversky_alpha=lc.get('tversky_alpha', 0.3), tversky_beta=lc.get('tversky_beta', 0.7),
    focal_gamma=lc.get('focal_gamma', 2.0),
)
print('Loss:', lc['type'])

In [ ]:
# === Training (transfer-learning 2-tahap + resume + grad-clip) — RE-RUNNABLE ===
import time
from forestwatch.training.trainer import TrainConfig, train

tcfg = TrainConfig(
    epochs=cfg['training']['epochs'], patience=cfg['training']['patience'],
    learning_rate=cfg['training']['learning_rate'], weight_decay=cfg['training']['weight_decay'],
    amp=cfg['training']['amp'], warmup_epochs=cfg['training'].get('warmup_epochs', 3),
    freeze_encoder_epochs=cfg['training'].get('freeze_encoder_epochs', 3),
    grad_clip=cfg['training'].get('grad_clip', 1.0), resume=cfg['training'].get('resume', True),
    seed=cfg['project']['seed'], ckpt_path=CKPT_PATH.as_posix(),
)
_t0 = time.time()
summary = train(model, train_loader, val_loader, loss_fn=loss_fn, cfg=tcfg)
train_minutes = round((time.time() - _t0) / 60, 1)
print(f"\nbest val mIoU = {summary['best_val_iou']:.4f} @ epoch {summary['best_epoch']} | {train_minutes} menit")
print("ckpt   :", summary['ckpt_path'])

In [ ]:
# === Plot history -> MODEL_DIR ===
import matplotlib.pyplot as plt
from forestwatch.constants import CLASS_COLORS

hist = summary['history']; epochs = [h['epoch'] for h in hist]
fig, axes = plt.subplots(1, 3, figsize=(17, 4))
axes[0].plot(epochs, [h['train_loss'] for h in hist], label='train', lw=2)
axes[0].plot(epochs, [h['val_loss'] for h in hist], label='val', lw=2)
axes[0].set_title('Loss'); axes[0].set_xlabel('Epoch'); axes[0].legend(); axes[0].grid(alpha=0.3)
axes[1].plot(epochs, [h['val_miou'] for h in hist], color='green', lw=2)
axes[1].axhline(0.60, color='orange', ls='--', label='min 0.60')
axes[1].axhline(0.75, color='red', ls='--', label='ideal 0.75')
axes[1].set_title('Val mIoU (makro)'); axes[1].set_ylim(0, 1); axes[1].legend(); axes[1].grid(alpha=0.3)
if hist and 'val_iou_per_class' in hist[-1]:
    for c in range(N_CLASSES):
        ys = [h.get('val_iou_per_class', [float('nan')]*N_CLASSES)[c] for h in hist]
        axes[2].plot(epochs, ys, color=CLASS_COLORS[c], label=CLASS_NAMES[c], lw=1.6)
    axes[2].set_title('Val IoU per-kelas'); axes[2].set_ylim(0, 1)
    axes[2].legend(fontsize=7, ncol=2); axes[2].grid(alpha=0.3)
else:
    axes[2].plot(epochs, [h['lr'] for h in hist], color='purple'); axes[2].set_title('LR')
fig.suptitle(f'{MODEL_KEY} — Merauke + Boven Digoel'); fig.tight_layout()
fig.savefig(MODEL_DIR / 'training_curve.png', dpi=120, bbox_inches='tight'); plt.show()
save_json({'best_val_iou': summary.get('best_val_iou'), 'best_epoch': summary.get('best_epoch'),
           'history': hist}, MODEL_DIR / 'training_history.json')
print('Disimpan:', MODEL_DIR / 'training_curve.png', '+ training_history.json')

In [ ]:
# === Evaluasi TEST (holdout Merauke+BD) -> metrics.json + confusion_matrix.png ===
import numpy as np, torch
import matplotlib.pyplot as plt
from forestwatch.training.trainer import evaluate
from forestwatch.training.metrics import compute_confusion_matrix, metric_summary

model.load_state_dict(torch.load(CKPT_PATH, map_location='cpu'))
preds, targets = evaluate(model, test_loader)
cm = compute_confusion_matrix(preds, targets, n_classes=N_CLASSES)
metrics = metric_summary(cm, class_names=CLASS_NAMES)
print(f"OA={metrics['overall_accuracy']*100:.2f}% | mIoU={metrics['mean_iou']:.4f} | kappa={metrics['kappa']:.4f}")
for row in metrics['per_class']:
    print(f"  {row['class']:<16} IoU={row['iou']:.4f}  F1={row['f1']:.4f}")
save_json(metrics, MODEL_DIR / 'metrics.json')

cm_np = np.array(metrics['confusion_matrix']); cm_norm = cm_np / cm_np.sum(axis=1, keepdims=True).clip(1)
fig, ax = plt.subplots(figsize=(8, 6)); im = ax.imshow(cm_norm, cmap='Blues', vmin=0, vmax=1)
for i in range(N_CLASSES):
    for j in range(N_CLASSES):
        ax.text(j, i, f'{cm_norm[i, j]:.2f}', ha='center', va='center', fontsize=9,
                color='white' if cm_norm[i, j] > 0.5 else 'black')
ax.set_xticks(range(N_CLASSES)); ax.set_xticklabels(CLASS_NAMES, rotation=45, ha='right')
ax.set_yticks(range(N_CLASSES)); ax.set_yticklabels(CLASS_NAMES)
ax.set_xlabel('Predicted'); ax.set_ylabel('True'); ax.set_title(f'Confusion — {MODEL_KEY} (Merauke+BD)')
fig.colorbar(im, ax=ax); fig.tight_layout()
fig.savefig(MODEL_DIR / 'confusion_matrix.png', dpi=120, bbox_inches='tight'); plt.show()
print('Disimpan:', MODEL_DIR / 'metrics.json', '+ confusion_matrix.png')

In [ ]:
# === Ekspor ONNX + tulis summary.json ===
from forestwatch.model.architecture import export_to_onnx
export_to_onnx(model, MODEL_DIR / 'model.onnx', in_channels=cfg['model']['in_channels'],
               patch_size=cfg['inference']['patch_size'], opset_version=13)

save_json({
    'model_key': MODEL_KEY, **MODEL_ARCH, 'area': 'Merauke + Boven Digoel',
    'tile_whitelist': sorted(TILE_WHITELIST),
    'best_val_iou': summary['best_val_iou'], 'best_epoch': summary['best_epoch'],
    'test_mean_iou': metrics['mean_iou'], 'test_overall_accuracy': metrics['overall_accuracy'],
    'test_kappa': metrics['kappa'], 'per_class': metrics['per_class'],
    'n_parameters': count_parameters(model), 'train_minutes': train_minutes,
    'n_train': len(final_train_files), 'n_val': len(val_files), 'n_test': len(test_files),
    'epochs_cfg': cfg['training']['epochs'], 'batch_size': cfg['training']['batch_size'],
    'class_weights': [float(w) for w in class_weights],
}, MODEL_DIR / 'summary.json')
print(f"OK Model 1 (Merauke+BD) SELESAI. Semua artefak di: {MODEL_DIR}")